# 02 - Entrenamiento TFT

**Temporal Fusion Transformer** - Estado del Arte en Series Temporales.

Usa tu RTX 3060 para entrenar. Tiempo estimado: 15-40 min.

In [ ]:
import torch
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Usando: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from pathlib import Path
DATA_DIR = Path('../data')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

# Cargar dataset de ruta (mas rapido para empezar; cambia a train_full.parquet para el modelo completo)
df = pd.read_parquet(DATA_DIR / 'train_route.parquet')
df['time_idx'] = (pd.to_datetime(df['date']) - pd.to_datetime(df['date']).min()).dt.days
df['id_station'] = df['id_station'].astype(str)

MAX_ENCODER = 60
MAX_PREDICTION = 7
training_cutoff = df['time_idx'].max() - 30
print(f'Datos: {df.shape} | Cutoff en idx={training_cutoff}')

In [ ]:
training = TimeSeriesDataSet(
    df[df['time_idx'] <= training_cutoff],
    time_idx='time_idx',
    target='price_gasolina_95_e5',
    group_ids=['id_station'],
    max_encoder_length=MAX_ENCODER,
    max_prediction_length=MAX_PREDICTION,
    static_categoricals=['id_station', 'name', 'municipality'],
    time_varying_known_reals=['time_idx', 'day_of_week', 'day_of_month', 'month', 'is_weekend'],
    time_varying_unknown_reals=[
        'price_gasolina_95_e5', 'price_lag_1d', 'price_lag_7d',
        'brent_usd', 'brent_eur', 'eur_usd',
        'brent_lag_7d', 'brent_lag_14d', 'brent_lag_21d', 'brent_lag_28d'
    ],
    target_normalizer=GroupNormalizer(groups=['id_station'], transformation='softplus'),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)
validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)
train_dl = training.to_dataloader(train=True, batch_size=64, num_workers=0)
val_dl = validation.to_dataloader(train=False, batch_size=64, num_workers=0)
print(f'Train batches: {len(train_dl)} | Val batches: {len(val_dl)}')

In [ ]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=3e-3,
    hidden_size=64,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=32,
    loss=QuantileLoss(),
    log_interval=10,
    optimizer='adam',
    reduce_on_plateau_patience=4,
)
print(f'Parametros del modelo: {tft.size():,}')

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, mode='min')
checkpoint = ModelCheckpoint(
    dirpath=str(MODELS_DIR),
    filename='tft-gaspredict-{epoch:02d}-{val_loss:.4f}',
    save_top_k=1, monitor='val_loss', mode='min'
)
trainer = L.Trainer(
    max_epochs=50,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stop, checkpoint],
    enable_progress_bar=True,
)
trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)
print(f'Modelo guardado en: {checkpoint.best_model_path}')